# 05 — Export Research Model Artifact

In [15]:
import json, joblib
from datetime import datetime
from pathlib import Path

from sklearn.preprocessing import StandardScaler, LabelEncoder
from notebook_utils import select_training_table, standardize_training_table, build_feature_table, find_project_root

In [16]:
raw_df, chosen = select_training_table()
df = standardize_training_table(raw_df)
x,y_text, FEATURE_NAMES = build_feature_table(df)

In [17]:
print("training rows:", len(x))
print("features:", len(FEATURE_NAMES))
print("features names:", FEATURE_NAMES)

training rows: 526
features: 17
features names: ['depth_mid_ft', 'depth_span_ft', 'porosity_mid_pct', 'porosity_span_pct', 'perm_mid_md', 'perm_span_md', 'api_mid', 'api_span', 'visc_mid_cp', 'visc_span_cp', 'so_mid_pct', 'so_span_pct', 'log10_perm_mid', 'log10_visc_mid', 'formation_sandstone', 'formation_carbonates', 'formation_unconsolidated_sands']


In [18]:
# ------------------------------------------------------------
# 2. Encode target labels
# ------------------------------------------------------------

le = LabelEncoder()
y = le.fit_transform(y_text)

print("\nClasses:")
for i, name in enumerate(le.classes_):
    print(f"  {i}: {name}")


Classes:
  0: Combustion
  1: HC immiscible
  2: Hot water
  3: Miscible CO2
  4: Miscible HC
  5: Miscible acid gas
  6: Nitrogen immiscible
  7: Polymer
  8: Steam


In [19]:
# ------------------------------------------------------------
# 3. Identify selected benchmark winner
# ------------------------------------------------------------

from notebook_utils import PROCESSED_DIR

selected_model_path = PROCESSED_DIR / "selected_model.json"
with open(selected_model_path, "r", encoding="utf-8") as f:
    selected_info = json.load(f)

selected = selected_info["best_model"]
print("\nSelected model:", selected)


Selected model: CatBoost


In [20]:
# ------------------------------------------------------------
# 4. Load fitted benchmark estimator
# ------------------------------------------------------------
from notebook_utils import safe_model_filename

candidate_path = (
    PROCESSED_DIR
    / f"model_{safe_model_filename(selected)}.joblib"
)

if not candidate_path.exists():
    raise FileNotFoundError(
        f"Could not find fitted model: {candidate_path}"
    )

candidate = joblib.load(candidate_path)

print("Loaded estimator:", type(candidate).__name__)

Loaded estimator: CatBoostClassifier


In [21]:
final_model = candidate.fit(x, y)

In [22]:
# ------------------------------------------------------------
# 6. Validate fitted artifact
# ------------------------------------------------------------

print("\nArtifact validation")
print("-" * 60)
print("Estimator type :", type(final_model).__name__)

if not hasattr(final_model, "predict_proba"):
    raise ValueError("Selected model does not support predict_proba().")

if hasattr(final_model, "n_features_in_"):
    print("Input features:", final_model.n_features_in_)

# For Pipeline, inspect the final classifier
if hasattr(final_model, "named_steps"):
    print("Pipeline steps:", list(final_model.named_steps.keys()))

    knn = final_model.named_steps.get("model")

    if knn is not None:
        print("KNN neighbors:",getattr(knn, "n_neighbors", None))
        print("KNN weights:",getattr(knn, "weights", None))
        print("KNN metric:", getattr(knn, "metric", None))




Artifact validation
------------------------------------------------------------
Estimator type : CatBoostClassifier
Input features: 17


In [23]:
# ------------------------------------------------------------
# 7. Export directory
# ------------------------------------------------------------
from notebook_utils import ARTIFACT_DIR
ARTIFACT_DIR.mkdir( parents=True, exist_ok=True)

In [24]:
# ------------------------------------------------------------
# 8. Export model pipeline
# ------------------------------------------------------------

MODEL_PATH = ( ARTIFACT_DIR / "model_v1.0.0.joblib")
joblib.dump( final_model, MODEL_PATH)

['c:\\Users\\mnabielizzuddin.radz\\OneDrive - PETRONAS\\Reservoir Engineering\\Programming_Python_Projects\\EOR ATLAS\\EORWEB\\EORWEBDEV\\outputs\\model_artifacts\\model_v1.0.0.joblib']

In [25]:
# ------------------------------------------------------------
# 9. Export label encoder
# ------------------------------------------------------------

ENCODER_PATH = ( ARTIFACT_DIR / "label_encoder_model_v1.0.0.joblib")
joblib.dump( le, ENCODER_PATH)


['c:\\Users\\mnabielizzuddin.radz\\OneDrive - PETRONAS\\Reservoir Engineering\\Programming_Python_Projects\\EOR ATLAS\\EORWEB\\EORWEBDEV\\outputs\\model_artifacts\\label_encoder_model_v1.0.0.joblib']

In [26]:
# ------------------------------------------------------------
# 10. Extract CatBoost configuration
# ------------------------------------------------------------

catboost_config = {}

# final_model should be the selected CatBoost model
# loaded from the benchmark and re-fitted on the full dataset.

catboost_config = {
    "iterations": getattr(
        final_model,
        "iterations",
        None
    ),
    "depth": getattr(
        final_model,
        "depth",
        None
    ),
    "learning_rate": getattr(
        final_model,
        "learning_rate",
        None
    ),
    "loss_function": getattr(
        final_model,
        "loss_function_",
        None
    ),
    "eval_metric": getattr(
        final_model,
        "eval_metric",
        None
    ),
    "random_seed": getattr(
        final_model,
        "random_seed_",
        None
    ),
    "l2_leaf_reg": getattr(
        final_model,
        "get_params",
        lambda: {}
    )().get("l2_leaf_reg", None),
    "random_strength": getattr(
        final_model,
        "get_params",
        lambda: {}
    )().get("random_strength", None),
    "border_count": getattr(
        final_model,
        "get_params",
        lambda: {}
    )().get("border_count", None),
}



In [27]:
# ------------------------------------------------------------
# 11. Validate CatBoost model
# ------------------------------------------------------------

print("Model type:", type(final_model).__name__)

if type(final_model).__name__ != "CatBoostClassifier":
    raise ValueError(
        "Selected model is not CatBoostClassifier. "
        f"Found: {type(final_model).__name__}"
    )

if not hasattr(final_model, "predict_proba"):
    raise ValueError(
        "CatBoost model does not support predict_proba()."
    )

print(
    "Feature count:",
    getattr(final_model, "n_features_in_", None)
)

print(
    "Classes:",
    list(le.classes_)
)


Model type: CatBoostClassifier
Feature count: 17
Classes: ['Combustion', 'HC immiscible', 'Hot water', 'Miscible CO2', 'Miscible HC', 'Miscible acid gas', 'Nitrogen immiscible', 'Polymer', 'Steam']


In [29]:
# ------------------------------------------------------------
# 12. Export configuration
# ------------------------------------------------------------

CONFIG_PATH = (
    ARTIFACT_DIR
    / "config_catboost_v1.0.0.json"
)

config = {
    "model_name": "EOR CatBoost",
    "version": "1.0.0",

    "algorithm": "CatBoostClassifier",
    "framework": "CatBoost",

    "pipeline": [
        "CatBoostClassifier"
    ],

    "input_schema_version": "EOR_ATLAS_ML_V2",

    "feature_count": len(FEATURE_NAMES),
    "feature_names": list(FEATURE_NAMES),

    "classes": list(le.classes_),

    "training_rows": len(x),
    "training_source": chosen,

    "fuzzy_used_as_model_feature": False,

    "selected_from_benchmark": True,

    "exported_at": datetime.now().isoformat(),

    "catboost": catboost_config,
}

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        indent=2,
        default=str,
    )

print(
    "\nConfiguration exported:",
    CONFIG_PATH
)




Configuration exported: c:\Users\mnabielizzuddin.radz\OneDrive - PETRONAS\Reservoir Engineering\Programming_Python_Projects\EOR ATLAS\EORWEB\EORWEBDEV\outputs\model_artifacts\config_catboost_v1.0.0.json


In [30]:
# ------------------------------------------------------------
# 13. Export CatBoost model
# ------------------------------------------------------------

MODEL_PATH = (
    ARTIFACT_DIR
    / "eor_catboost_v1.0.0.joblib"
)

joblib.dump(
    final_model,
    MODEL_PATH
)

print(
    "Model exported:",
    MODEL_PATH
)



Model exported: c:\Users\mnabielizzuddin.radz\OneDrive - PETRONAS\Reservoir Engineering\Programming_Python_Projects\EOR ATLAS\EORWEB\EORWEBDEV\outputs\model_artifacts\eor_catboost_v1.0.0.joblib


In [31]:
# ------------------------------------------------------------
# 14. Export label encoder
# ------------------------------------------------------------

ENCODER_PATH = (
    ARTIFACT_DIR
    / "label_encoder_catboost_v1.0.0.joblib"
)

joblib.dump(
    le,
    ENCODER_PATH
)

print(
    "Label encoder exported:",
    ENCODER_PATH
)



Label encoder exported: c:\Users\mnabielizzuddin.radz\OneDrive - PETRONAS\Reservoir Engineering\Programming_Python_Projects\EOR ATLAS\EORWEB\EORWEBDEV\outputs\model_artifacts\label_encoder_catboost_v1.0.0.joblib


In [32]:
# ------------------------------------------------------------
# 15. Export manifest
# ------------------------------------------------------------

MANIFEST_PATH = (
    ARTIFACT_DIR
    / "model_manifest_catboost_v1.0.0.json"
)

manifest = {
    "version": "1.0.0",
    "model_name": "EOR CatBoost",
    "algorithm": "CatBoostClassifier",

    "model_path": str(MODEL_PATH),
    "encoder_path": str(ENCODER_PATH),
    "config_path": str(CONFIG_PATH),

    "feature_count": len(FEATURE_NAMES),
    "class_count": len(le.classes_),

    "created": datetime.now().isoformat(),
}

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        default=str,
    )

print(
    "Manifest exported:",
    MANIFEST_PATH
)


Manifest exported: c:\Users\mnabielizzuddin.radz\OneDrive - PETRONAS\Reservoir Engineering\Programming_Python_Projects\EOR ATLAS\EORWEB\EORWEBDEV\outputs\model_artifacts\model_manifest_catboost_v1.0.0.json


In [33]:
# ------------------------------------------------------------
# 16. Final export summary
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("CATBOOST ARTIFACT EXPORT COMPLETE")
print("=" * 70)

print("Model   :", MODEL_PATH)
print("Encoder :", ENCODER_PATH)
print("Config  :", CONFIG_PATH)
print("Manifest:", MANIFEST_PATH)

print("\nModel:")
print(type(final_model).__name__)

print("\nFeatures:")
print(len(FEATURE_NAMES))

print("\nClasses:")
for i, cls in enumerate(le.classes_):
    print(f"  {i}: {cls}")



CATBOOST ARTIFACT EXPORT COMPLETE
Model   : c:\Users\mnabielizzuddin.radz\OneDrive - PETRONAS\Reservoir Engineering\Programming_Python_Projects\EOR ATLAS\EORWEB\EORWEBDEV\outputs\model_artifacts\eor_catboost_v1.0.0.joblib
Encoder : c:\Users\mnabielizzuddin.radz\OneDrive - PETRONAS\Reservoir Engineering\Programming_Python_Projects\EOR ATLAS\EORWEB\EORWEBDEV\outputs\model_artifacts\label_encoder_catboost_v1.0.0.joblib
Config  : c:\Users\mnabielizzuddin.radz\OneDrive - PETRONAS\Reservoir Engineering\Programming_Python_Projects\EOR ATLAS\EORWEB\EORWEBDEV\outputs\model_artifacts\config_catboost_v1.0.0.json
Manifest: c:\Users\mnabielizzuddin.radz\OneDrive - PETRONAS\Reservoir Engineering\Programming_Python_Projects\EOR ATLAS\EORWEB\EORWEBDEV\outputs\model_artifacts\model_manifest_catboost_v1.0.0.json

Model:
CatBoostClassifier

Features:
17

Classes:
  0: Combustion
  1: HC immiscible
  2: Hot water
  3: Miscible CO2
  4: Miscible HC
  5: Miscible acid gas
  6: Nitrogen immiscible
  7: Pol